# 01. Build 1 km grid features

Constructs the 1 km prediction grid over the study area (northern and
central Germany, lon 9.4-15.9, lat 50.0-53.8, EPSG:3035) and computes
all features the trained BayesNF expects, for every day in 2010-2023.

Static features (sampled once): coords, elevation (Copernicus DEM),
six SoilGrids variables.
Dynamic features (per day): IDW, GOS, SVD-quantiles svd_00..svd_04 using
the full train-station rainfall pool as neighbour set.

Output: monthly parquets under
`s3://thesis-data-ismaktam/dataset/grid_1km/YYYY-MM.parquet`.
Schema is identical to `bayesnf_holdout_test.parquet`.


## 1. Setup


In [1]:
import os
import sys
import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

ROOT = Path('../..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

from thesis.config import Config
from thesis.data.dem import DEMSource
from thesis.data.rekis import ReKISSource
from thesis.data.soilgrids import SoilGridsSource
from thesis.grid import build_prediction_grid
from thesis.models.grk.features import compute_day_geo_features

N_CORES = os.cpu_count() or 1
print(f'cwd       : {os.getcwd()}')
print(f'CPU cores : {N_CORES}')


cwd       : /root/precip_interpolation_thesis
CPU cores : 256


## 2. Config

`K_GEO` and `SVD_QUANTILES` must match the values used to build the
training parquets (`src/thesis/scripts/run_grk_kfold_cv.py`), otherwise
the `idw`, `gos` and `svd_*` columns mean something different to the
trained model. Training used `K_GEO=15` and 21 evenly-spaced quantiles
`np.arange(0.0, 1.05, 0.05)` producing `svd_00..svd_20`. The trained
BayesNF only reads `svd_00..svd_04`, but `gos` depends on the full
21-dim quantile profile so all 21 must be computed.


In [2]:
GRID_RESOLUTION_M = 1000
DATE_START        = '2023-01-01'
DATE_END          = '2023-12-31'

K_GEO         = 15
SVD_QUANTILES = np.arange(0.0, 1.05, 0.05)
assert len(SVD_QUANTILES) == 21

# Only these svd_* columns are read by the trained BayesNF; the rest
# (svd_05..svd_20) are computed for gos consistency but dropped on save.
KEEP_SVD = ['svd_00', 'svd_01', 'svd_02', 'svd_03', 'svd_04']

SOIL_VARS = ('bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa')

OUT_LOCAL  = Path('results/dataset/grid_1km')
OUT_LOCAL.mkdir(parents=True, exist_ok=True)

S3_BUCKET = 'thesis-data-ismaktam'
S3_PREFIX = 'dataset/grid_1km'

SKIP_EXISTING = True

print(f'resolution     : {GRID_RESOLUTION_M} m')
print(f'period         : {DATE_START} -> {DATE_END}')
print(f'k_geo          : {K_GEO}  svd_quantiles: {SVD_QUANTILES.tolist()}')
print(f'output (local) : {OUT_LOCAL}')
print(f'output (s3)    : s3://{S3_BUCKET}/{S3_PREFIX}/YYYY-MM.parquet')


resolution     : 1000 m
period         : 2023-01-01 -> 2023-12-31
k_geo          : 15  svd_quantiles: [0.0, 0.05, 0.1, 0.15000000000000002, 0.2, 0.25, 0.30000000000000004, 0.35000000000000003, 0.4, 0.45, 0.5, 0.55, 0.6000000000000001, 0.65, 0.7000000000000001, 0.75, 0.8, 0.8500000000000001, 0.9, 0.9500000000000001, 1.0]
output (local) : results/dataset/grid_1km
output (s3)    : s3://thesis-data-ismaktam/dataset/grid_1km/YYYY-MM.parquet


## 3. Build 1 km grid + static features


In [3]:
cfg = Config()
cfg.study_area.grid_resolution_m = GRID_RESOLUTION_M

dem = DEMSource(cfg)

t0 = time.time()
grid = build_prediction_grid(cfg, dem=dem)
print(f'grid built in {time.time()-t0:.1f}s')
print(f'  shape (H, W) : {grid.shape}')
print(f'  n_cells      : {grid.coords_proj.shape[0]:,}')

from pyproj import Transformer
tx = Transformer.from_crs(cfg.study_area.target_crs, 'EPSG:4326', always_xy=True)
lon, lat = tx.transform(grid.coords_proj[:, 0], grid.coords_proj[:, 1])

static = pd.DataFrame({
    'x_proj'     : grid.coords_proj[:, 0].astype(np.float64),
    'y_proj'     : grid.coords_proj[:, 1].astype(np.float64),
    'longitude'  : lon.astype(np.float64),
    'latitude'   : lat.astype(np.float64),
    'elevation_m': grid.elevation_m.astype(np.float64),
})
static['cell_id'] = np.arange(len(static), dtype=np.int64)

for var in SOIL_VARS:
    src = SoilGridsSource(cfg, variable=var)
    static[var] = src.sample_at_points(lon, lat).astype(np.float32)
    print(f'  {var:14s} mean={static[var].mean():.2f}  nan={static[var].isna().sum()}')

# Drop ocean / out-of-coverage cells (NaN soil or DEM)
n_before = len(static)
static = static.dropna().reset_index(drop=True)
static['cell_id'] = np.arange(len(static), dtype=np.int64)
print(f'cells after NaN drop: {len(static):,}  (was {n_before:,})')

STATIC_PATH = OUT_LOCAL.parent / 'grid_1km_static.parquet'
static.to_parquet(STATIC_PATH, index=False)
print(f'static features saved -> {STATIC_PATH}  ({STATIC_PATH.stat().st_size/1e6:.1f} MB)')


grid built in 0.3s
  shape (H, W) : (462, 439)
  n_cells      : 202,818
  bulk_density   mean=124.58  nan=0
  clay           mean=167.53  nan=0
  sand           mean=451.24  nan=0
  silt           mean=309.48  nan=0
  soc            mean=301.42  nan=0
  water_10kpa    mean=-413.77  nan=0
cells after NaN drop: 202,818  (was 202,818)
static features saved -> results/dataset/grid_1km_static.parquet  (13.9 MB)


## 4. Train-station rainfall pool


In [4]:
rekis = ReKISSource(cfg)
rain = rekis.load(DATE_START, DATE_END, exclude_holdout=True)
rain = rain.dropna(subset=['precip_mm']).reset_index(drop=True)

stn_xy = (
    rain[['station_id', 'lon', 'lat']]
    .drop_duplicates('station_id')
    .reset_index(drop=True)
)
tx_fwd = Transformer.from_crs('EPSG:4326', cfg.study_area.target_crs, always_xy=True)
stn_xy['x_proj'], stn_xy['y_proj'] = tx_fwd.transform(
    stn_xy['lon'].values, stn_xy['lat'].values,
)
stn_xy_map = stn_xy.set_index('station_id')[['x_proj', 'y_proj']]

rain = rain.merge(stn_xy_map, on='station_id', how='left')
rain['date'] = pd.to_datetime(rain['date'])

print(f'train stations : {stn_xy["station_id"].nunique()}')
print(f'station-days   : {len(rain):,}')
print(f'date range     : {rain["date"].min().date()} -> {rain["date"].max().date()}')


train stations : 1966
station-days   : 717,590
date range     : 2023-01-01 -> 2023-12-31


## 5. Per-day FFRK features at grid cells

For each day, `compute_day_geo_features` is called with:

* `xy_all` = train stations concatenated with grid cell centers,
* `z_all` = train rainfall concatenated with zeros (unused for grid rows),
* `train_mask` = True for stations, False for grid cells.

Only grid-cell records are kept. Per-month batches are written to
parquet locally, then uploaded to S3.


In [5]:
import boto3
s3 = boto3.client('s3')

def _s3_key_exists(key):
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=key)
        return True
    except s3.exceptions.ClientError:
        return False

# Top-level grid arrays (module-scope so joblib/loky memory-maps them
# across all worker processes instead of re-pickling per task).
grid_xy_full = static[['x_proj', 'y_proj']].to_numpy().astype(np.float64)
grid_ids     = static['cell_id'].to_numpy()
n_grid       = len(grid_xy_full)
grid_sids    = np.array([f'grid_{i:08d}' for i in grid_ids])

# Worker function — picklable, takes all arrays as explicit args.
def _compute_one_day(date, train_xy, train_z, train_sids,
                     grid_xy, grid_sids, k_geo, svd_q):
    n_train = len(train_xy)
    if n_train < k_geo:
        return None
    xy_all   = np.vstack([train_xy, grid_xy])
    z_all    = np.concatenate([train_z, np.zeros(len(grid_xy))])
    sids_all = np.concatenate([train_sids, grid_sids])
    mask     = np.concatenate([np.ones(n_train, bool), np.zeros(len(grid_xy), bool)])
    records = compute_day_geo_features(date, xy_all, z_all, sids_all, mask,
                                        k_geo, svd_q)
    return [r for r in records if r['station_id'].startswith('grid_')]


# Plan months upfront: check S3 for already-done months, skip them.
all_months = list(pd.period_range(DATE_START, DATE_END, freq='M'))
months_to_do = []
for month in all_months:
    out_name = f'{month.year:04d}-{month.month:02d}.parquet'
    s3_key = f'{S3_PREFIX}/{out_name}'
    if SKIP_EXISTING and _s3_key_exists(s3_key):
        print(f'  skip {out_name} (already in s3)')
        continue
    months_to_do.append(month)
print(f'months to process: {len(months_to_do)} / {len(all_months)}')

# Build the FULL task list across all months at once. This is the
# crucial change vs the previous per-month Parallel(): joblib now sees
# thousands of tasks and can saturate every core, instead of 30-day
# batches that only feed a fraction of N_CORES.
day_args = []
day_month = []
for month in months_to_do:
    m_start = max(pd.Timestamp(f'{month.year}-{month.month:02d}-01'),
                  pd.Timestamp(DATE_START))
    m_end = min(pd.Timestamp(month.end_time.date()), pd.Timestamp(DATE_END))
    month_rain = rain[(rain['date'] >= m_start) & (rain['date'] <= m_end)]
    by_date = {d: g for d, g in month_rain.groupby('date')}
    for d in pd.date_range(m_start, m_end, freq='D'):
        if d not in by_date:
            continue
        g = by_date[d]
        day_args.append((
            str(d.date()),
            g[['x_proj', 'y_proj']].to_numpy().astype(np.float64),
            g['precip_mm'].to_numpy().astype(np.float64),
            g['station_id'].to_numpy(),
        ))
        day_month.append(f'{month.year:04d}-{month.month:02d}')

print(f'total day-tasks  : {len(day_args):,}')
print(f'will run on      : {N_CORES} cores via joblib/loky')

t_total = time.time()
results = Parallel(n_jobs=N_CORES, backend='loky',
                   verbose=10, pre_dispatch='2*n_jobs',
                   batch_size='auto')(
    delayed(_compute_one_day)(date, txy, tz, tsids,
                              grid_xy_full, grid_sids, K_GEO, SVD_QUANTILES)
    for date, txy, tz, tsids in day_args
)
print(f'\nFFRK compute elapsed: {(time.time()-t_total)/60:.1f} min')

# Group results back by month and write parquets sequentially.
from collections import defaultdict
by_month = defaultdict(list)
for month_key, recs in zip(day_month, results):
    if recs:
        by_month[month_key].extend(recs)

for month_key, recs in sorted(by_month.items()):
    df_m = pd.DataFrame(recs)
    df_m['date'] = pd.to_datetime(df_m['date'])
    df_m = df_m.rename(columns={'date': 'datetime', 'station_id': 'cell_sid'})
    df_m['cell_id'] = df_m['cell_sid'].str.removeprefix('grid_').astype(np.int64)
    df_m = df_m.drop(columns=['cell_sid']).merge(static, on='cell_id', how='left')

    # Drop unused svd_05..svd_20 (computed for gos consistency only).
    drop_svd = [f'svd_{i:02d}' for i in range(5, 21) if f'svd_{i:02d}' in df_m.columns]
    if drop_svd:
        df_m = df_m.drop(columns=drop_svd)

    for c in ('idw', 'gos', *KEEP_SVD):
        df_m[c] = df_m[c].astype(np.float32)
    df_m['rainfall']     = np.float64('nan')
    df_m['rainfall_int'] = np.int32(-1)
    df_m['station_id']   = df_m['cell_id'].astype(str)
    df_m['fold']         = np.int8(-2)

    col_order = [
        'datetime', 'latitude', 'longitude', 'x_proj', 'y_proj', 'elevation_m',
        'idw', 'gos', 'svd_00', 'svd_01', 'svd_02', 'svd_03', 'svd_04',
        'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa',
        'rainfall', 'rainfall_int', 'station_id', 'cell_id', 'fold',
    ]
    df_m = df_m[col_order]

    out_local = OUT_LOCAL / f'{month_key}.parquet'
    df_m.to_parquet(out_local, index=False)
    s3.upload_file(str(out_local), S3_BUCKET, f'{S3_PREFIX}/{month_key}.parquet')
    print(f'  {month_key}: {len(df_m):>10,} rows  '
          f'({out_local.stat().st_size/1e6:.0f} MB)')

print(f'\ntotal elapsed: {(time.time()-t_total)/60:.1f} min')


months to process: 12 / 12
total day-tasks  : 365
will run on      : 256 cores via joblib/loky


[Parallel(n_jobs=256)]: Using backend LokyBackend with 256 concurrent workers.
[Parallel(n_jobs=256)]: Done   2 out of 365 | elapsed: 12.9min remaining: 2336.1min
[Parallel(n_jobs=256)]: Done  39 out of 365 | elapsed: 13.6min remaining: 113.4min
[Parallel(n_jobs=256)]: Done  76 out of 365 | elapsed: 14.1min remaining: 53.8min
[Parallel(n_jobs=256)]: Done 113 out of 365 | elapsed: 14.8min remaining: 33.0min
[Parallel(n_jobs=256)]: Done 150 out of 365 | elapsed: 15.5min remaining: 22.2min
[Parallel(n_jobs=256)]: Done 187 out of 365 | elapsed: 16.2min remaining: 15.4min
[Parallel(n_jobs=256)]: Done 224 out of 365 | elapsed: 16.9min remaining: 10.6min
[Parallel(n_jobs=256)]: Done 261 out of 365 | elapsed: 18.7min remaining:  7.5min
[Parallel(n_jobs=256)]: Done 298 out of 365 | elapsed: 19.5min remaining:  4.4min
[Parallel(n_jobs=256)]: Done 335 out of 365 | elapsed: 20.2min remaining:  1.8min
[Parallel(n_jobs=256)]: Done 365 out of 365 | elapsed: 20.8min finished



FFRK compute elapsed: 20.8 min
  2023-01:  6,287,358 rows  (358 MB)
  2023-02:  5,678,904 rows  (319 MB)
  2023-03:  6,287,358 rows  (359 MB)
  2023-04:  6,084,540 rows  (341 MB)
  2023-05:  6,287,358 rows  (337 MB)
  2023-06:  6,084,540 rows  (335 MB)
  2023-07:  6,287,358 rows  (353 MB)
  2023-08:  6,287,358 rows  (355 MB)
  2023-09:  6,084,540 rows  (327 MB)
  2023-10:  6,287,358 rows  (361 MB)
  2023-11:  6,084,540 rows  (353 MB)
  2023-12:  6,287,358 rows  (360 MB)

total elapsed: 31.4 min
